## C5_02 — Construirea vector store-ului pentru o bulă
În acest notebook construim un vector store FAISS pentru o singură bulă / un singur agent.
Fiecare student lucrează pe bula lui. Scopul este să vedem clar cum textele curățate devin embeddings, apoi index FAISS.
Mai târziu, aceeași logică va fi pusă într-un script `.py` care rulează automat pentru toate bulele.

## 0. Setup

In [15]:
pip install sentence_transformers

  Using cached pyyaml-6.0.3-cp313-cp313-win_amd64.whl.metadata (2.4 kB)
  Using cached tokenizers-0.22.2-cp39-abi3-win_amd64.whl.metadata (7.4 kB)
  Using cached safetensors-0.7.0-cp38-abi3-win_amd64.whl.metadata (4.2 kB)
  Using cached click-8.3.3-py3-none-any.whl.metadata (2.6 kB)
  Using cached shellingham-1.5.4-py2.py3-none-any.whl.metadata (3.5 kB)
  Using cached rich-15.0.0-py3-none-any.whl.metadata (18 kB)
  Using cached annotated_doc-0.0.4-py3-none-any.whl.metadata (6.6 kB)
  Using cached mdurl-0.1.2-py3-none-any.whl.metadata (1.6 kB)
   ---------------------------------------- 0.0/588.7 kB ? eta -:--:--
   ---------------------------------------- 0.0/588.7 kB ? eta -:--:--
   ----------------- ---------------------- 262.1/588.7 kB ? eta -:--:--
   ---------------------------------------- 588.7/588.7 kB 1.5 MB/s  0:00:00
   ---------------------------------------- 0.0/10.6 MB ? eta -:--:--
    --------------------------------------- 0.3/10.6 MB ? eta -:--:--
   -- -------------

In [16]:
from pathlib import Path
import os, pickle
import pandas as pd
import faiss
from sentence_transformers import SentenceTransformer

while not Path("data/bubbles").exists():
    os.chdir("..")

BUBBLES_DIR = Path("data/bubbles")
VECTOR_DIR = Path("assets/vectorstores")
VECTOR_DIR.mkdir(parents=True, exist_ok=True)

MODEL_NAME = "paraphrase-multilingual-MiniLM-L12-v2"

c:\Users\nasta\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Aleg bula mea
Alege fișierul `.jsonl` al bulei tale.
Acest fișier a fost creat în etapa anterioară, după verificarea manuală a textelor.

In [17]:
MY_BUBBLE_FILE = "pro_european.jsonl" 

bubble_path = BUBBLES_DIR / MY_BUBBLE_FILE
slug = bubble_path.stem

df_bubble = pd.read_json(bubble_path, lines=True)

print("Bula:", slug)
print("Texte:", len(df_bubble))

df_bubble[["id", "agent", "text"]].head()

Bula: pro_european
Texte: 50


,id,agent,text
0,yt_yEuctxNb4O0_UgxTCkwcP96Sb5_Rpht4AaABAg,Pro-european,Tipul care și-a dat demisia în noiembrie la cu...
1,yt_kLdQS9N5cBU_UgxIxee_zETaR-Dlu894AaABAg,Pro-european,"+ la asta, d. Președinte, trebuie sa existe O ..."
2,yt_FAe_Sqr2vfU_UgzfkQjlhFS0o3ZaCMl4AaABAg,Pro-european,Mă bucur să văd un Președinte care se comportă...
3,yt_Fcmw2yFzz8I_UgxuNvJZTDz3pFm5Ijh4AaABAg,Pro-european,Deci exemplu asta cu Parisul cred ca e cea mai...
4,yt_XlCX1l_6feU_UgwBIpp5ZbEtJkybe3t4AaABAg,Pro-european,"meritocratie in politica inseamna, totusi, alt..."


## 2. Pregătim textele
Pentru FAISS avem nevoie de o listă simplă de texte.
Metadata rămâne separat, ca să putem lega fiecare vector de textul original.

In [18]:
texts = df_bubble["text"].fillna("").tolist()
metadata = df_bubble.to_dict(orient="records")

print("Primul text:")
print(texts[0][:500])

Primul text:
Tipul care și-a dat demisia în noiembrie la cum gândește și vorbește merita sa fie ministrul justiției. Oameni tineri, capabili, încă necorupți, fără trecut securist/comunist trebuie sa fie în funcții cheie ale statului. Vezi miniștrii USR, maxim 40 de ani, pregătiți, implicați....


## 3. Generăm embeddings
Un embedding este o reprezentare vectorială a textului: texte apropiate ca sens primesc vectori apropiați în spațiul semantic.
Folosim un model multilingv, deoarece corpusul este în limba română.
Normalizăm vectorii la lungime 1, astfel încât produsul scalar din FAISS să funcționeze ca similaritate cosinus.

In [19]:
model = SentenceTransformer(MODEL_NAME)
embeddings = model.encode(
    texts,
    normalize_embeddings=True,
    show_progress_bar=True
).astype("float32")
print("Număr texte:", len(texts))
print("Dimensiune embeddings:", embeddings.shape)

c:\Users\nasta\AppData\Local\Programs\Python\Python313\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\nasta\.cache\huggingface\hub\models--sentence-transformers--paraphrase-multilingual-MiniLM-L12-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Batches: 100%|██████████| 2/2 [00:04<00:00,  2.13s/it]

Număr texte: 50
Dimensiune embeddings: (50, 384)


### Verificare rapidă
Răspunde în 1–2 propoziții în notebook:
- Câte texte are bula ta?
- Câți vectori au fost generați?
- Ce înseamnă a doua valoare din `embeddings.shape`?

In [ ]:
# TODO student:
# Bula mea are 50 texte.
# Au fost generați 50 vectori.
# A doua valoare din embeddings.shape reprezintă numărul de vectori pentru fiecare text.

## 4. Construim indexul FAISS
FAISS este biblioteca care caută rapid vectori apropiați.
Indexul nu păstrează textele originale. El păstrează doar reprezentările vectoriale.
De aceea salvăm două lucruri:
- `index.faiss` = indexul vectorial;
- `index.pkl` = textele originale și metadatele.

In [20]:
index = faiss.IndexFlatIP(embeddings.shape[1])
index.add(embeddings)
out_dir = VECTOR_DIR / slug
out_dir.mkdir(parents=True, exist_ok=True)
faiss.write_index(index, str(out_dir / "index.faiss"))
with open(out_dir / "index.pkl", "wb") as f:
    pickle.dump(metadata, f)
print("Salvat în:", out_dir)
print("Vectori în index:", index.ntotal)

Salvat în: assets\vectorstores\pro_european
Vectori în index: 50


## 5. Verificăm fișierele create
Dacă totul a mers corect, bula ta are acum un folder propriu în `assets/vectorstores/`.
Acest folder trebuie să conțină `index.faiss` și `index.pkl`.

In [ ]:
# TODO student:
# index.faiss există: DA
# index.pkl există: DA
# index.ntotal este egal cu numărul de texte: ...

## Ce am construit?
Am transformat textele curate ale unei bule într-un index vectorial local.
Acest index nu generează răspunsuri. El doar permite căutarea semantică.
În continuare vom testa dacă, pentru o întrebare, FAISS returnează texte relevante.

## 6. Testăm retrieval-ul
Acum simulăm logica aplicației.
- Utilizatorul introduce o știre sau o afirmație politică.
- Retriever-ul caută în memoria bulei cele mai asemănătoare texte.
- Nu generăm încă un răspuns cu LLM. Doar verificăm ce exemple sunt recuperate.

In [27]:
# Text nou introdus în aplicație

input_text = "Uniunea Europeană reprezintă un avantaj de dezvoltare pentru România"

In [28]:
# Transformăm textul nou în embedding

query_vector = model.encode(
    [input_text],
    normalize_embeddings=True
).astype("float32")

In [11]:
# query_vector

In [29]:
# Căutăm cele mai apropiate 5 texte din bula noastră

scores, results = index.search(query_vector, k=5)

for rank, pos in enumerate(results[0], start=1):
    row = metadata[pos]
    
    print(f"\nRezultat {rank}")
    print("Scor:", round(float(scores[0][rank-1]), 3))
    print("Text:", row["text"][:500])


Rezultat 1
Scor: 0.77
Text: Visul Romaniei de la 1848 a fost sa faca politica Europeana. Sa fie inclusa in Europa si sa fie Europa. Avem acum acest lucru! Ne-am îndeplinit visul iar asta duce la o bunastare fantastica (Romania este cea mai prospera din istorie!) Si exista unii trepanati da ne spuna ca UE nu e nimic. Va dati seama!? Nu zic ca nu mai avem treaba. Mai avem enorm de mult. Dar avem si posibilitatea sa criticam guvernul. Ceea ce ex-EU nu permite. Acolo te “aliniezi” cu interesul national!

Rezultat 2
Scor: 0.634
Text: + la asta, d. Președinte, trebuie sa existe O MAJORITATE a cetatenilor si in Romănia..., care sa doreasca UNIREA....

Rezultat 3
Scor: 0.611
Text: ❤❤ NICUȘOR DAN președinte ♥️ MULȚUMIM UE Schengen NATO România Republica Moldova Ucraina ♥️

Rezultat 4
Scor: 0.587
Text: Unde este stegul Uniunii Europene si steagul NATO din imaginea de la Cotroceni?!...eu pt astea două steaguri împreună cu al României am votat 😠...să va fie ruşine România nu e nimeni si nimic far

### TODO
Schimbă `input_text` cu o afirmație potrivită pentru agentul tău.
Rulează căutarea.
Notează:
- câte rezultate din 5 sunt relevante; toate
- dacă textele recuperate exprimă vocea agentului; da
- dacă ai observat un text slab care ar trebui eliminat; nu